## Electrostatic problem

In [167]:
from ngsolve import *
from netgen.read_gmsh import ReadGmsh
from ngsolve.webgui import Draw
from netgen.csg import *
import math
import pyvista as pv
import numpy as np
from ngsolve.krylovspace import GMRes

mesh_path = '../meshes/coil_box'
output_path = '../output/case1/case1_ngsolve'

# Import geometries
mesh = ReadGmsh(mesh_path + ".msh")

for i in range(1, 3):
    # print(i)
    mesh.SetMaterial(i, f'{i}')

for i in range(1, 13):
    # print(i)
    mesh.SetBCName(i-1, f'{i}')

mesh = Mesh(mesh)

mesh.ngmesh.Save(mesh_path + ".vol")

In [168]:
mesh.ne, mesh.nv, mesh.GetMaterials(), mesh.GetBoundaries()

(69629,
 12090,
 ('1', '2'),
 ('1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12'))

In [169]:
# Define material parameters

# f = 123.5e3  # Frequency in Hz
I_coil = 5000.0 # A
mu0 = 4*math.pi*1e-7
# omega = 2*math.pi*f # f Hz excitation

sigma = {"1": 1.0, "2": 5.998e7}  # Electric conductivity [S/m]
# sigma = {"1": 1.0, "2": 1.0e-7}  # Electric conductivity [S/m]
mu_r = {"1": 1.0, "2": 1.0} # Relative permeability [-]

mu_cf = mu0 * CoefficientFunction([mu_r.get(mat, 1.0) for mat in mesh.GetMaterials()])
sigma_cf = CoefficientFunction([sigma.get(mat, 0.0) for mat in mesh.GetMaterials()])

In [170]:
crosssection = Integrate(1, mesh, definedon=mesh.Boundaries("11"))

fespot = H1(mesh, order=1, definedon=mesh.Materials("2"), dirichlet="8")
phi,psi = fespot.TnT()
with TaskManager():
    bfa = BilinearForm(sigma_cf*grad(phi)*grad(psi)*dx).Assemble()
    inv = bfa.mat.Inverse(freedofs=fespot.FreeDofs(), inverse="sparsecholesky")
    lff = LinearForm(I_coil/crosssection*psi*ds("11")).Assemble()
    gfphi = GridFunction(fespot)
    gfphi.vec.data = inv * lff.vec

In [171]:
# fes = HCurl(mesh, order=1, complex=True, dirichlet="VacuumSurface", gradientdomains="Tile|Block|Pipe")
# fes = HCurl(mesh, order=1, complex=True, dirichlet="1|2|3|4|5|12", nograds = False)
fes = HCurl(mesh, order=1, complex=True, dirichlet="1|2|3|4|5|12", nograds = False)
print ("HCurl dofs:", fes.ndof)
u,v = fes.TnT()

a = BilinearForm(fes, symmetric=True, condense=True)
a += 1/mu_cf*curl(u)*curl(v)*dx+1e-7/mu_cf*u*v*dx

pre = preconditioners.BDDC(a)
f = LinearForm(sigma_cf*grad(gfphi)*v*dx("2"))

A = GridFunction(fes)

# with TaskManager():
#     solvers.BVP(bf=a, lf=f, gf=A, pre=pre, \
#                 solver=solvers.CGSolver, solver_flags={"plotrates": True, "tol" : 1e-6})
    

a.Assemble()
f.Assemble()
# pre = preconditioners.Local(a)
with TaskManager():
    A.vec.data = GMRes(a.mat, f.vec, pre=pre, maxsteps=100, printrates=True, tol= 1e-6)

HCurl dofs: 165218
GMRes iteration 1, residual = 75578648.42502707     
GMRes iteration 2, residual = 40507281.405259304     
GMRes iteration 3, residual = 22100020.926416036     
GMRes iteration 4, residual = 15642270.81263584     
GMRes iteration 5, residual = 8493947.987739768     
GMRes iteration 6, residual = 4951938.849790819     
GMRes iteration 7, residual = 2745172.5293863397     
GMRes iteration 8, residual = 1519050.998059252     
GMRes iteration 9, residual = 873229.7284656501     
GMRes iteration 10, residual = 462641.3193105556     
GMRes iteration 11, residual = 252851.25112751036     
GMRes iteration 12, residual = 141784.84454760907     
GMRes iteration 13, residual = 82006.79456710095     
GMRes iteration 14, residual = 46769.79691866945     
GMRes iteration 15, residual = 26464.6619175443     
GMRes iteration 16, residual = 15500.817943149075     
GMRes iteration 17, residual = 8808.597817124806     
GMRes iteration 18, residual = 4819.4996671260715     
GMRes iterat

In [172]:
B_cf = curl(A)

fes_B = VectorH1(mesh, order=1)
B = GridFunction(fes_B)
B.Set(B_cf.real)

B_norm = Norm(B)

# B_norm = Norm(B_cf)

In [173]:
res = pv.read(mesh_path + ".msh")
points = res.points

B_sol = np.zeros((points.shape[0], 3))

B_norm_sol = np.zeros((points.shape[0], 1))

for i in range(points.shape[0]):
    point = mesh(points[i, 0], points[i, 1], points[i, 2])
    B_sol[i, :] = B(point)
    B_norm_sol[i, :] = B_norm(point)

res["B"] = B_sol
res["Magnetic_flux_density_norm"] = B_norm_sol

res.save(output_path + ".vtu")

In [5]:
import sys
import gmsh

gmsh.initialize()

# Load your existing mesh
gmsh.open('../meshes/coil_box.msh')

# Define your mapping: {Material_ID: "Name"}
# In Gmsh: dim=3 for volumes, dim=2 for surfaces
material_mapping = {
    1: "vacuum",
    2: "wire"
}

# Apply the names
for mat_id, name in material_mapping.items():
    # Find the tag of the entity with this ID. 
    # Usually, if you created the mesh with ID 2, the tag is 2.
    gmsh.model.addPhysicalGroup(dim=3, tags=[mat_id], tag=mat_id, name=name)

gmsh.model.addPhysicalGroup(dim=2, tags=[8], tag=8, name="CoilIn")
gmsh.model.addPhysicalGroup(dim=2, tags=[11], tag=11, name="CoilOut")

# Save the mesh back out
gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
gmsh.write('../meshes/coil_box_named.msh')

gmsh.finalize()

Info    : Reading '../meshes/coil_box.msh'...
Info    : 12090 nodes
Info    : 75923 elements
Info    : Done reading '../meshes/coil_box.msh'
Info    : Writing '../meshes/coil_box_named.msh'...
Info    : Done writing '../meshes/coil_box_named.msh'
